<span style="margin: 0; font-size: 25px; font-weight: 500">Short-Latency Hand Movement Classification based on Surface-EMG Spectrogram with PCA</span>
<br>
<span style="margin: 0; font-size: 16px">Xiaolong Zhai, Beth Jelfs, Rosa H. M. Chan, Chung Tin</span>

<span style="margin: 0; font-size: 15px;">implemented for</span>
<br>
<span style="margin: 0; font-size: 18px;">Research Paper Implementation</span>
<br>
<span style="margin: 0; font-size: 15px;">BM4152 - Bio-Signal Processing</span>
<br>

<span style="margin: 0; font-size: 15px;">by Fernando D. S. (220163X) and Udugamasooriya P. H. J. (220658U)</span>

### 1. Introduction and Preliminaries

In this paper, the authors make use of the Ninapro DB2 dataset. It contains EMG data collected from 40 subjects performing 49 different hand movements.

The 49 movements are put into three broad categories as follows:
1. E1 - basic movements of fingers (and wrist)
2. E2 - grasping and functional movements
3. E3 - force patterns

The entire DB2 dataset may be accessed [here](https://ninapro.hevs.ch/instructions/DB2.html). The page contains 40 `.zip` files, named `s1.zip` through `s40.zip`, each one containing the data collected from a single subject.
Each `.zip` file contains three files, named `SX_E1_A1.mat`, `SX_E2_A1.mat` and `SX_E3_A1.mat`, where `X` is the subject index. The second portion of the filename identifies the category of movements contained within it.

In this notebook, we will only utilize 17 movements in category E1 (18 including rest). We will inspect the contents of `S2_E1_A1.mat` to understand how the data is structured.

In [1]:
import numpy as np
import scipy
import sklearn
import random
import pymatreader # to read .mat files

subject_data = pymatreader.read_mat('../DB2_s2/S2_E1_A1.mat')

In [105]:
print (subject_data.keys())

dict_keys(['__header__', '__version__', '__globals__', 'emg', 'acc', 'stimulus', 'glove', 'inclin', 'subject', 'exercise', 'repetition', 'restimulus', 'rerepetition'])


In [106]:
print (
    subject_data['emg'].shape,
    subject_data['stimulus'].shape,
    subject_data['repetition'].shape,
    subject_data['restimulus'].shape,
    subject_data['rerepetition'].shape
)

print (
    np.unique(subject_data['stimulus']),
    np.unique(subject_data['repetition'])
)

(1803711, 12) (1803711,) (1803711,) (1803711,) (1803711,)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17] [0 1 2 3 4 5 6]


`subject_data['emg']` is a $1803711 \times 12$ matrix. Each column corresponds to a single EMG channel, and each row is a single sample. The sampling frequency is given to be $2\text{ kHz}$.

`subject_data['stimulus']` identifies each sample with the particular movement being performed by the subject, and is a number between 0 and 17. `subject_data['repetition']` identifies each sample with which repetition of the corresponding movement it is associated with, and is a number between 0 and 6 (0 being a neutral value, and 1 through 6 identifying the first through sixth repetitions).

`subject_data['restimulus']` and `subject_data['rerepetition']` are similar, but contain relabeled values, by accounting for delays and discrepancies that the subject may have shown in copying the instructions given.

In [107]:
fs = 2000 # sampling frequency

# Construct an array with strings identifying each movement index
movement_names = [
    "rest",
    "thumb up",
    "extension of index and middle, flexion of others",
    "flexion of ring and pinky, extension of others",
    "thumb opposing base of pinky",
    "abduction of all fingers",
    "fingers flexed together in fist",
    "pointing index",
    "adduction of extended fingers",
    "wrist supination (axis: middle finger)",
    "wrist pronation (axis: middle finger)",
    "wrist supination (axis: little finger)",
    "wrist pronation (axis: little finger)",
    "wrist flexion",
    "wrist extension",
    "wrist radial deviation",
    "wrist ulnar deviation",
    "wrist extension with closed hand"
]

We will implement a function to obtain the entire EMG signal corresponding to a particular repetition of a particular movement, from each of the 12 channels, from a specified subject. Because there are disproportionately many repetitions of the 'rest' movement, we implement a different function to randomly extract rest signals instead, to keep the data balanced (as described in the paper).

In [ ]:
def extract_rest_indices(subject_data, num_repetitions, random_seed):
    """
        Randomly extracts the start and end indices of where the specified number of rest movements can be found, and modifies subject_data by adding new keys.
    
        Parameters
        ----------
        subject_data : dict
            dictionary returned by pymatreader after reading the relevant .mat file corresponding to a subject

        num_repetitions : number
            extracts this many repetitions of the rest movement
        
        random_seed : number
            seed for a random number generator to pick random rest samples, ideally subject-specific

        Returns
        -------
        None
    """

    random.seed(random_seed)

    for prefix in ['re', '']:
        movement_ids = subject_data[prefix + 'stimulus']

        # All samples where movement_ids == 0 correspond to a rest
        rests = (movement_ids == 0).astype(int)

        # Pad the beginning and end with 0s (we begin and end with a non-rest, so any rest, if present, must be strictly inside the array), to eliminate edge cases
        rests_padded = np.pad(rests, pad_width=1, mode='constant', constant_values=0)

        # Note we drop the first and last index, to avoid possible long rest margins at the beginning and end
        rest_start_indices = np.where(np.diff(rests_padded) == 1)[0][1 : -1]
        rest_end_indices = np.where(np.diff(rests_padded) == -1)[0][1 : -1]

        random_indices = random.sample(range(len(rest_start_indices)), num_repetitions)

        subject_data[prefix + 'reststart'] = rest_start_indices[random_indices]
        subject_data[prefix + 'restend'] = rest_end_indices[random_indices]

def emg_of_movement(subject_data, movement_id, repetition_num, corrected=True):
    """
        Extracts the samples corresponding to a specified movement index and repetition number from all 12 channels.
    
        Parameters
        ----------
        subject_data : dict
            dictionary returned by pymatreader after reading the relevant .mat file corresponding to a subject

        movement_id : number
            between 0 and 17, identifying the movement

        repetition_num : number
            between 1 and 6, identifying which repetition in particular to extract
    
        corrected : boolean
            if True, uses the revised labels to extract the signal; otherwise uses raw labels
        
        Returns
        -------
        n x 12 array
            where n is the number of samples associated with the specified repetition of the specified movement, and the second dimension corresponds to each channel
            
    """
    
    if (movement_id < 0) or (movement_id > 17):
        print ("Invalid Movement ID")
        return

    if (repetition_num < 1) or (repetition_num > 6):
        print ("Invalid Repetition Number")
        return

    emg_data = subject_data['emg']

    # Extract rests separately
    if (movement_id == 0):
        if corrected:
            start_index = subject_data['rereststart'][repetition_num - 1]
            end_index = subject_data['rerestend'][repetition_num - 1]
        else:
            start_index = subject_data['reststart'][repetition_num - 1]
            end_index = subject_data['restend'][repetition_num - 1]
        return emg_data[start_index : end_index, :]

    if corrected:
        movement_ids = subject_data['restimulus']
        repetition_nums = subject_data['rerepetition']
    else:
        movement_ids = subject_data['stimulus']
        repetition_nums = subject_data['repetition']
        
    return emg_data[(movement_ids == movement_id) & (repetition_nums == repetition_num), :]

We will implement another function to construct segments from a provided set of EMG samples from each channel.

In [109]:
def construct_segments(emg_data, num_samples, step_samples):
    """
        Constructs segments of consecutive samples from the provided EMG data from all (12) channels.
    
        Parameters
        ----------
        emg_data : array
            an n x N array, n samples obtained from each of N channels (the dataset here has N = 12 channels in total)

        num_samples : number
            number of consecutive samples to include in a segment from each channel
    
        step_samples : number
            number of samples to skip forward from the start of the current segment, to form the next
        
        Returns
        -------
        n x N array
            where n is the number of samples associated with the specified repetition of the specified movement, and the second dimension corresponds to each channel
    """

    n, _ = emg_data.shape # total number of samples per channel

    if (num_samples > n) or (num_samples <= 0):
        print ("Invalid Number of Samples")
        return []

    if (step_samples <= 0):
        print ("Invalid Step")
        return []
    
    segments = []

    for i in range(0, n - num_samples + 1, step_samples):
        segments.append(emg_data[i : i + num_samples, :])

    return segments

Next, we implement a function to compute the normalized spectrogram of a provided signal segment from each channel. We also implement a function to compute the raw, un-normalized spectrogram, for use in training to determine the normalizing limits.

In [110]:
def normalized_spectrograms_vector(segment, normalizer, upper_cutoff_freq_index, **spectrogram_kwargs):
    """
        Computes the normalized spectrogram of the signal segment provided from each channel, and returns a vector with all spectrograms concatenated.

        Parameters
        ----------
        segment : array
            an n x N array, where n is the number of samples in the segment and N is the number of channels

        normalizer : function
            a function to normalize the spectrogram computed per channel

        upper_cutoff_freq_index : number
            only the frequency bins starting from index 0 to this index will be retained from the spectrogram computed
            
        **spectrogram_kwargs : dictionary
            keyword arguments required by scipy.signal.spectrogram
    
        Returns
        -------
        array
            a flattened and normalized vector of all spectrograms concatenated, whose dimensions depend on the cutoff frequency and other parameters set for scipy.signal.spectrogram
    """
    
    spectrogram_vectors = []
    channel_index = 0

    for channel in segment.T:
        _, _, spectrogram = scipy.signal.spectrogram(channel, **spectrogram_kwargs)

        # Retain only the specified frequency bins
        spectrogram = spectrogram[:upper_cutoff_freq_index, :]

        # Normalize and flatten to a vector
        spectrogram_vector = normalizer(spectrogram.flatten(), channel_index)
        channel_index += 1

        spectrogram_vectors.append(spectrogram_vector)

    return np.concatenate(spectrogram_vectors)

def raw_spectrogram_vector(segment, upper_cutoff_freq_index, **spectrogram_kwargs):
    """
        Computes the spectrogram of the signal segment provided from each channel and returns a flattened vector for each channel.

        Parameters
        ----------
        segment : array
            an n x N array, where n is the number of samples in the segment and N is the number of channels

        upper_cutoff_freq_index : number
            only the frequency bins starting from index 0 to this index will be retained from the spectrogram computed
            
        **spectrogram_kwargs : dictionary
            keyword arguments required by scipy.signal.spectrogram
    
        Returns
        -------
        array
            a list of vectors, each vector being the flattened spectrogram computed for a channel
    """

    spectrogram_vectors = []

    for channel in segment.T:
        _, _, spectrogram = scipy.signal.spectrogram(channel, **spectrogram_kwargs)

        # Retain only the specified frequency bins, and flatten to a vector
        spectrogram = spectrogram[:upper_cutoff_freq_index, :].flatten()

        spectrogram_vectors.append(spectrogram)

    return spectrogram_vectors

To compare the performance of using the spectrogram for classification, we also implement a function to compute the RMS of a signal segment provided from each channel, and attempt to classify using RMS as well.

In [111]:
def RMS(segment):
    """
        Computes the RMS of the signal segment provided from each channel.

        Parameters
        ----------
        segment : array
            an n x N array, where n is the number of samples in the segment and N is the number of channels
    
        Returns
        -------
        1 x N array
            containing the RMS value of the segment from each channel
            
    """

    return np.array([
        np.sqrt(np.mean(channel ** 2)) for channel in segment.T
    ])

### 2. Training and Testing

We will now implement functions to prepare data for training and testing a PCA transformer and classifier.

We train with the 1<sup>st</sup>, 3<sup>rd</sup>, 4<sup>th</sup>, and 6<sup>th</sup> repetitions of each movement by each user, and test on the remaining repetitions.

First, we construct the training and testing data sets.

In [ ]:
def spectrogram_dataset_for_subject(subject_id):
    """
        Extracts data from the specified subject and returns a training and testing data set constructed using spectrograms, along with a function for normalizing data.

        Parameters
        ----------
        subject_id : number
            between 1 and 40, identifying the subject
    
        Returns
        -------
        (normalizer, X_train, y_train, X_test, y_test) : tuple
            - normalizer:   function to which a raw flattened spectrogram from a channel can be passed to normalize it
            - X_train:      list of spectrogram vectors for training
            - y_train:      list of labels identifying the movement class corresponding to each entry in X_train
            - X_test:       similar to X_train, but with test data
            - y_test:       similar to y_train, but with test labels
    """
    
    X_train_raw, X_train, y_train = [], [], []
    X_test_raw, X_test, y_test = [], [], []

    subject_data = pymatreader.read_mat(f'../DB2_s{subject_id}/S{subject_id}_E1_A1.mat')
    extract_rest_indices(subject_data, 6, subject_id + 5)
    
    for movement_id in range(0, 18):
        for repetition_num in range(1, 7):

            full_emg_data = emg_of_movement(subject_data, movement_id, repetition_num)
            segments = construct_segments(full_emg_data, 400, 200)

            # For now, we append per-channel flattened spectrograms. Later we will normalize and concatenate them.
            if (repetition_num == 2 or repetition_num == 5):
                for segment in segments:
                    X_test_raw.append(raw_spectrogram_vector(
                        segment, 95,
                        fs=fs,
                        window='hamming',
                        nperseg=256,
                        noverlap=184,
                        nfft=256,
                        detrend=False
                    ))
                    y_test.append(movement_id)
            else:
                for segment in segments:
                    X_train_raw.append(raw_spectrogram_vector(
                        segment, 95,
                        fs=fs,
                        window='hamming',
                        nperseg=256,
                        noverlap=184,
                        nfft=256,
                        detrend=False
                    ))
                    y_train.append(movement_id)

    X_train_raw = np.array(X_train_raw)

    # Compute 1st and 99th percentiles for normalization as described 
    num_train_segments, num_channels, _ = X_train_raw.shape
    min_values = [np.percentile(X_train_raw[:, i, :], 1) for i in range(num_channels)]
    max_values = [np.percentile(X_train_raw[:, i, :], 99) for i in range(num_channels)]

    # Define a function for normalization, taking care to clip values that land outside the percentiles
    def normalizer(spectrogram_vector, channel_index):
        """
            Normalizes (and clips to 0 and 1) the data in a flattened spectrogram from a given channel.
    
            Parameters
            ----------
            spectrogram_vector : array
                a flattened vector spectrogram vector obtained from a single channel

            channel_index : number
                a number identifying the channel from which the spectrogram has been computed
        
            Returns
            -------
            vector
                the same flat spectrogram vector, but with its entries scaled and clipped to lie within 0 and 1
        """
        
        scaled = (spectrogram_vector - min_values[channel_index]) / (max_values[channel_index] - min_values[channel_index])
        return np.clip(scaled, 0, 1)

    # Prepare the data in the format expected for training, by normalizing each spectrogram and concatenating them
    for segment_index in range(num_train_segments):
        train_vector = []

        for channel_index in range(num_channels):
            spectrogram_vector = X_train_raw[segment_index, channel_index, :]

            train_vector.append(normalizer(spectrogram_vector, channel_index))

        X_train.append(np.concatenate(train_vector))

    X_train = np.array(X_train)
    y_train = np.array(y_train)

    # Repeat with test data
    X_test_raw = np.array(X_test_raw)
    num_test_segments, _, _ = X_test_raw.shape

    for segment_index in range(num_test_segments):
        test_vector = []

        for channel_index in range(num_channels):
            spectrogram_vector = X_test_raw[segment_index, channel_index, :]

            test_vector.append(normalizer(spectrogram_vector, channel_index))

        X_test.append(np.concatenate(test_vector))

    X_test = np.array(X_test)
    y_test = np.array(y_test)

    return (normalizer, X_train, y_train, X_test, y_test)

In [113]:
normalizer, X_train, y_train, X_test, y_test = spectrogram_dataset_for_subject(2)

In [114]:
print (X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(3085, 3420) (3085,) (1508, 3420) (1508,)


Now, we train a PCA transformer with the training data.

In [117]:
pca = sklearn.decomposition.PCA(n_components=100)
X_train_reduced = pca.fit_transform(X_train)

Then, we perform a grid-search with 4-fold cross-validation to choose the best parameters $c$ and $\gamma$ for an SVM classifier for the data.

In [ ]:
parameters = {
    'C': [2**i for i in range(-2, 15)],
    'gamma': [2**i for i in range(-12, 8)]
} # As the authors have described

grid_search = sklearn.model_selection.GridSearchCV(
    sklearn.svm.SVC(kernel='rbf'),
    parameters, cv=4, scoring='accuracy'
)

grid_search.fit(X_train_reduced, y_train)

Best Parameters Found: {'C': 4, 'gamma': 0.015625}
Best CV Score: 0.6302073882919027


In [131]:
svm_best_params = grid_search.best_params_
print("Best Parameters Found:", svm_best_params)
print("Best CV Score:", grid_search.best_score_)

# Train an SVM with the best parameters found
svm = sklearn.svm.SVC(kernel='rbf', C=svm_best_params['C'], gamma=svm_best_params['gamma'])
svm.fit(X_train_reduced, y_train)

Best Parameters Found: {'C': 4, 'gamma': 0.015625}
Best CV Score: 0.6302073882919027


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",4
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",0.015625
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide <shrinking_svm>`.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide <scores_probabilities>`...deprecated:: 1.9 The `probability` parameter is deprecated and will be removed in 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`.",'deprecated'
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
